# Voyexa AI — Model 4: Geo-Clustering for Day-wise Itineraries
Given a set of places a user wants to visit, cluster them geographically into "day trip" groups so the itinerary reads "Day 1: these 3 nearby places, Day 2: these 4 others" instead of zig-zagging across the map.

Uses KMeans on lat/long — `num_days` forces exact k, or leave it out to auto-pick k via silhouette score (best for exploratory grouping, but tends to favor fewer, well-separated clusters).

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

DATA_PATH = os.path.join("..", "..", "data", "processed", "places_processed.csv")
df = pd.read_csv(DATA_PATH)
df.shape

(1163, 202)

## Clustering function

In [2]:
def cluster_places_for_itinerary(place_ids, num_days=None):
    subset = df[df["place_id"].isin(place_ids)].copy()
    if len(subset) < 2:
        subset["day"] = 1
        return subset

    coords = subset[["Latitude", "Longitude"]].values

    if num_days is not None:
        k = min(num_days, len(subset))
    else:
        best_k, best_score = 2, -1
        max_k = min(6, len(subset) - 1)
        for k in range(2, max_k + 1):
            labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(coords)
            if len(set(labels)) < 2:
                continue
            score = silhouette_score(coords, labels)
            if score > best_score:
                best_score, best_k = score, k
        k = best_k

    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    subset["day"] = km.fit_predict(coords) + 1

    centroid_order = subset.groupby("day")[["Latitude", "Longitude"]].mean().sort_values("Latitude").index
    day_map = {old: new for new, old in enumerate(centroid_order, start=1)}
    subset["day"] = subset["day"].map(day_map)

    return subset.sort_values(["day"])

## Demo: 4-day itinerary for a Jammu & Kashmir shortlist

In [3]:
jk_ids = df[df["State"] == "Jammu And Kashmir"]["place_id"].tolist()
print(f"Total candidate places in state: {len(jk_ids)}")

result = cluster_places_for_itinerary(jk_ids[:15], num_days=4)
for day in sorted(result["day"].unique()):
    day_places = result[result["day"] == day]
    print(f"Day {day}: {', '.join(day_places['Place_Name'].tolist())} ({day_places['City'].iloc[0]})")

Total candidate places in state: 34
Day 1: Dogra Art Museum, Mubarak Mandi Palace, Amar Mahal Palace, Patnitop, Vaishno Devi Temple (Jammu)
Day 2: Gulmarg, Noori Chamb Waterfall (Baramulla)
Day 3: Shalimar Bagh, Dal Lake, Wular Lake, Manasbal (Srinagar)
Day 4: Sonamarg, Naranag, Pahalgam, Amarnath Cave Temple (Ganderbal)


## Demo: auto-k clustering (no fixed num_days)

In [4]:
result_auto = cluster_places_for_itinerary(jk_ids[:15])
print(f"Auto-selected {result_auto['day'].nunique()} day clusters")
for day in sorted(result_auto["day"].unique()):
    day_places = result_auto[result_auto["day"] == day]
    print(f"Day {day}: {', '.join(day_places['Place_Name'].tolist())}")

Auto-selected 2 day clusters
Day 1: Dogra Art Museum, Mubarak Mandi Palace, Amar Mahal Palace, Patnitop, Vaishno Devi Temple
Day 2: Dal Lake, Sonamarg, Shalimar Bagh, Wular Lake, Manasbal, Naranag, Noori Chamb Waterfall, Gulmarg, Pahalgam, Amarnath Cave Temple
